# Analysis of results

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)

WORKSPACE = Path(os.environ.get(
    "CELLBRIDGE_WORKSPACE",
    "/lustre/scratch126/cellgen/lotfollahi/so25",
))
RESULTS_ROOT = Path(os.environ.get(
    "CELLBRIDGE_INFERENCE_RESULTS",
    WORKSPACE / "outputs/reproducibility/inference",
))

ID_DATASETS = ["hest"]
OOD_DATASETS = ["lizard", "nucls", "pannuke", "phenocell"]
DATASET_LABELS = {
    "hest": "HEST",
    "lizard": "Lizard",
    "nucls": "NuCLS",
    "pannuke": "PanNuke",
    "phenocell": "PhenoCell",
}

BASELINE_LABELS = {
    "uni": "UNI",
    "virchow2": "Virchow2",
    "hibou_l": "Hibou-L",
    "lemon": "LEMON",
    "past": "PAST",
    "volta": "VOLTA",
    "spatialwhisperer": "SpatialWhisperer",
}
BASELINE_VERSIONS = {method: "all" for method in BASELINE_LABELS}
BASELINE_VERSIONS["lemon"] = "vitb8"
EXPECTED_METHODS = list(BASELINE_LABELS.values()) + [
    f"CellBridge {size} ({arm})"
    for size in ("5K", "10K", "50K", "all")
    for arm in ("adapted", "projected")
]

CLASSIFICATION_METRICS = ["macro_f1", "balanced_accuracy", "macro_auroc"]
PROBES = ["linear", "knn"]
TASK = "all"  # Change to "immune" for the PhenoCell immune-only task.
HEST_COMMON_SAMPLES = True  # Compare methods on their shared HEST sections.

RESULTS_ROOT

## Load results

A run is expected at `<method>/<version>/<dataset>` for OOD and `<method>/<version>/hest/<sample>` for separately submitted HEST jobs. Baseline versions are normally `all`; LEMON ViT-B/8 uses `vitb8`; CellBridge versions are `cells_5k`, `cells_10k`, `cells_50k`, and `cells_all`.

In [ ]:
def _require_columns(table, columns, path):
    missing = set(columns) - set(table.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {sorted(missing)}")


def _normalise_id(value):
    return str(value).strip().lower().replace("-", "_")


def paper_method(method, version, arm):
    method_id = _normalise_id(method)
    arm_id = _normalise_id(arm)
    version_id = _normalise_id(version)
    if method_id == "cellbridge" or arm_id.startswith("lora_"):
        size = version_id.removeprefix("cells_")
        size = size.upper() if size != "all" else "all"
        representation = arm_id.removeprefix("lora_")
        return f"CellBridge {size} ({representation})"

    label = BASELINE_LABELS.get(method_id, str(method))
    if version_id != BASELINE_VERSIONS.get(method_id, "all"):
        label = f"{label} [{version}]"
    if arm_id not in {method_id, "default"}:
        label = f"{label} ({arm})"
    return label


def iter_runs(root):
    if not root.exists():
        raise FileNotFoundError(f"Results root does not exist: {root}")
    for method_dir in sorted(path for path in root.iterdir() if path.is_dir()):
        for version_dir in sorted(path for path in method_dir.iterdir() if path.is_dir()):
            for dataset_dir in sorted(path for path in version_dir.iterdir() if path.is_dir()):
                dataset = dataset_dir.name
                if (dataset_dir / "evaluation").is_dir():
                    yield method_dir.name, version_dir.name, dataset, None, dataset_dir
                elif dataset == "hest":
                    for sample_dir in sorted(path for path in dataset_dir.iterdir() if path.is_dir()):
                        if (sample_dir / "evaluation").is_dir():
                            yield (
                                method_dir.name, version_dir.name, dataset,
                                sample_dir.name, sample_dir,
                            )


def _contextualise(table, path, method, version, dataset, sample_id, arm, task, probe):
    table = table.copy()
    context = {
        "method": method, "version": version, "dataset": dataset,
        "arm": arm, "task": task, "probe": probe,
    }
    for column, expected in context.items():
        if column in table and not (table[column].astype(str) == str(expected)).all():
            raise ValueError(f"{path}: {column} disagrees with its directory ({expected!r})")
        table[column] = expected
    if sample_id is not None:
        if "sample_id" in table and not (table["sample_id"].astype(str) == sample_id).all():
            raise ValueError(f"{path}: sample_id disagrees with its directory ({sample_id!r})")
        table["sample_id"] = sample_id
    elif "sample_id" not in table:
        table["sample_id"] = pd.NA
    table["paper_method"] = paper_method(method, version, arm)
    table["source_path"] = str(path)
    table["result_id"] = str(path.parent)
    return table


def load_results(root):
    classification = []
    uncertainty = []
    conformal = []
    empty_runs = []
    metric_columns = {
        "dataset", "method", "arm", "task", "probe",
        "accuracy", "balanced_accuracy", "macro_f1",
        "weighted_f1", "macro_auroc",
    }
    conformal_columns = {
        "dataset", "method", "arm", "task", "probe",
        "conformal_method", "marginal_coverage", "coverage_gap",
        "mean_prediction_set_size", "singleton_rate",
        "root_inconclusive_rate",
    }

    for method, version, dataset, sample_id, run_dir in iter_runs(root):
        metric_paths = sorted(run_dir.glob("evaluation/*/*/*/metrics.csv"))
        if not metric_paths:
            empty_runs.append(str(run_dir))
        for path in metric_paths:
            arm, task, probe, _ = path.relative_to(run_dir / "evaluation").parts
            table = pd.read_csv(path)
            _require_columns(table, metric_columns, path)
            table = _contextualise(
                table, path, method, version, dataset, sample_id, arm, task, probe
            )
            classification.append(table)

            uncertainty_path = path.with_name("uncertainty.csv")
            if uncertainty_path.exists():
                errors = pd.read_csv(uncertainty_path)
                _require_columns(errors, {"metric", "estimate", "bootstrap_se"}, uncertainty_path)
                errors = _contextualise(
                    errors, uncertainty_path, method, version, dataset,
                    sample_id, arm, task, probe,
                )
                uncertainty.append(errors)

        conformal_paths = sorted(
            run_dir.glob("evaluation/*/*/*/conformal_prediction/metrics.csv")
        )
        for path in conformal_paths:
            arm, task, probe, _, _ = path.relative_to(run_dir / "evaluation").parts
            table = pd.read_csv(path)
            _require_columns(table, conformal_columns, path)
            table = _contextualise(
                table, path, method, version, dataset, sample_id, arm, task, probe
            )
            conformal.append(table)

    if not classification:
        raise FileNotFoundError(f"No canonical evaluation metrics found below {root}")
    classification = pd.concat(classification, ignore_index=True)
    uncertainty = pd.concat(uncertainty, ignore_index=True) if uncertainty else pd.DataFrame()
    conformal = pd.concat(conformal, ignore_index=True) if conformal else pd.DataFrame()

    classification_key = [
        "dataset", "method", "version", "arm", "task", "probe", "sample_id"
    ]
    duplicates = classification.duplicated(classification_key, keep=False)
    if duplicates.any():
        columns = classification_key + ["source_path"]
        raise ValueError(
            "Duplicate classification results found:\n"
            + classification.loc[duplicates, columns].to_string(index=False)
        )
    if not conformal.empty:
        conformal_key = classification_key + ["conformal_method"]
        duplicates = conformal.duplicated(conformal_key, keep=False)
        if duplicates.any():
            columns = conformal_key + ["source_path"]
            raise ValueError(
                "Duplicate conformal results found:\n"
                + conformal.loc[duplicates, columns].to_string(index=False)
            )
    return classification, uncertainty, conformal, empty_runs


In [ ]:
classification, uncertainty, conformal, empty_runs = load_results(RESULTS_ROOT)

observed_methods = sorted(set(classification["paper_method"]) | set(conformal.get("paper_method", [])))
METHOD_ORDER = EXPECTED_METHODS + [name for name in observed_methods if name not in EXPECTED_METHODS]

hest_groups = (
    classification.loc[(classification["dataset"] == "hest") & (classification["task"] == TASK)]
    .groupby(["paper_method", "probe"])["sample_id"]
    .agg(lambda values: set(values.dropna().astype(str)))
)
HEST_SAMPLES = (
    sorted(set.intersection(*hest_groups))
    if HEST_COMMON_SAMPLES and len(hest_groups)
    else None
)
if HEST_SAMPLES is not None:
    print(f"Using {len(HEST_SAMPLES)} common HEST sections: {', '.join(HEST_SAMPLES)}")

print(f"Loaded {len(classification):,} classification rows and {len(conformal):,} conformal rows.")
if empty_runs:
    print(f"{len(empty_runs)} discovered run directories do not yet contain canonical metrics.")
    display(pd.DataFrame({"run_without_metrics": empty_runs}))

result_counts = (
    classification.loc[classification["task"] == TASK]
    .groupby(["paper_method", "dataset", "probe"], dropna=False)
    .size()
    .unstack(["dataset", "probe"], fill_value=0)
    .reindex(METHOD_ORDER, fill_value=0)
)
result_counts.index.name = "Method"
display(result_counts)

## Table helpers

Classification tables show HEST mean $\pm$ section SD. OOD tables show estimate $\pm$ grouped-bootstrap SE when the corresponding `uncertainty.csv` exists. AUROC has no bootstrap row and is therefore shown as a point estimate.

In [ ]:
def _format_value(value, error=np.nan, digits=3):
    if pd.isna(value):
        return "--"
    if pd.notna(error):
        return f"{value:.{digits}f} $\\pm$ {error:.{digits}f}"
    return f"{value:.{digits}f}"


def _method_dataset_grid(rows):
    rows = rows.reindex(METHOD_ORDER)
    rows.index.name = "Method"
    return rows


def classification_table(datasets, metric, probe, task="all", digits=3):
    if metric not in classification.columns:
        raise KeyError(f"Unknown classification metric: {metric}")
    selected = classification.loc[
        classification["dataset"].isin(datasets)
        & (classification["task"] == task)
        & (classification["probe"] == probe)
    ].copy()
    if HEST_SAMPLES is not None:
        selected = selected.loc[
            (selected["dataset"] != "hest")
            | selected["sample_id"].astype("string").isin(HEST_SAMPLES)
        ]
    error_lookup = {}
    if not uncertainty.empty:
        selected_errors = uncertainty.loc[
            uncertainty["dataset"].isin(datasets)
            & (uncertainty["task"] == task)
            & (uncertainty["probe"] == probe)
            & (uncertainty["metric"] == metric)
        ]
        error_lookup = (
            selected_errors.set_index(["paper_method", "dataset"])["bootstrap_se"].to_dict()
        )

    values = {}
    for (method, dataset), group in selected.groupby(["paper_method", "dataset"], sort=False):
        if dataset == "hest":
            value = group[metric].mean()
            error = group[metric].std(ddof=1)
        else:
            if len(group) != 1:
                raise ValueError(f"Expected one OOD result for {(method, dataset, task, probe)}, found {len(group)}")
            value = group[metric].iloc[0]
            error = error_lookup.get((method, dataset), np.nan)
        values[(method, DATASET_LABELS.get(dataset, dataset))] = _format_value(value, error, digits)

    columns = [DATASET_LABELS.get(dataset, dataset) for dataset in datasets]
    table = pd.Series(values, dtype=object).unstack() if values else pd.DataFrame()
    return _method_dataset_grid(table.reindex(columns=columns, fill_value="--").fillna("--"))


def emit_latex(table, caption, label):
    display(table)
    print(table.to_latex(
        escape=False, na_rep="--", caption=caption, label=label,
        position="t", column_format="l" + "c" * len(table.columns),
    ))

## ID classification tables (HEST)

In [ ]:
for probe in PROBES:
    for metric in CLASSIFICATION_METRICS:
        table = classification_table(ID_DATASETS, metric, probe, task=TASK)
        emit_latex(
            table,
            caption=f"ID {metric.replace('_', ' ')} using the {probe} probe.",
            label=f"tab:id-{metric.replace('_', '-')}-{probe}",
        )

## OOD classification tables

In [ ]:
for probe in PROBES:
    for metric in CLASSIFICATION_METRICS:
        table = classification_table(OOD_DATASETS, metric, probe, task=TASK)
        emit_latex(
            table,
            caption=f"OOD {metric.replace('_', ' ')} using the {probe} probe.",
            label=f"tab:ood-{metric.replace('_', '-')}-{probe}",
        )

## Conformal prediction tables

Each conformal variant gets one table with datasets grouped across columns. HEST cells are mean $\pm$ section SD; OOD cells are point estimates because conformal bootstrap uncertainty is not currently written. `coverage_gap` is in percentage points, as defined by the evaluation code.

In [ ]:
CONFORMAL_METHODS = [
    "split", "graph",
    "class_split", "class_graph",
    "branch_split", "branch_graph",
    "hybrid_split", "hybrid_graph",
]
CONFORMAL_METRICS = {
    "marginal_coverage": "Marginal coverage",
    "coverage_gap": "CovGap",
    "mean_prediction_set_size": "Mean set size",
    "singleton_rate": "Singleton rate",
}
CONFORMAL_PROBES = ["linear", "knn"]


def conformal_table(conformal_method, probe, task="all", digits=3):
    if conformal.empty:
        raise FileNotFoundError("No conformal metrics were discovered.")
    datasets = ID_DATASETS + OOD_DATASETS
    selected = conformal.loc[
        conformal["dataset"].isin(datasets)
        & (conformal["task"] == task)
        & (conformal["probe"] == probe)
        & (conformal["conformal_method"] == conformal_method)
    ].copy()

    if HEST_SAMPLES is not None:
        selected = selected.loc[
            (selected["dataset"] != "hest")
            | selected["sample_id"].astype("string").isin(HEST_SAMPLES)
        ]

    values = {}
    for (method, dataset), group in selected.groupby(["paper_method", "dataset"], sort=False):
        if dataset != "hest" and len(group) != 1:
            raise ValueError(
                f"Expected one OOD conformal result for {(method, dataset, task, probe, conformal_method)}, "
                f"found {len(group)}"
            )
        for metric, metric_label in CONFORMAL_METRICS.items():
            value = group[metric].mean()
            error = group[metric].std(ddof=1) if dataset == "hest" else np.nan
            column = (DATASET_LABELS.get(dataset, dataset), metric_label)
            values[(method, column)] = _format_value(value, error, digits)

    columns = pd.MultiIndex.from_product([
        [DATASET_LABELS.get(dataset, dataset) for dataset in datasets],
        list(CONFORMAL_METRICS.values()),
    ])
    if values:
        table = pd.Series(values, dtype=object).unstack()
    else:
        table = pd.DataFrame(columns=columns)
    table = table.reindex(columns=columns, fill_value="--").fillna("--")
    return _method_dataset_grid(table)


In [ ]:
for probe in CONFORMAL_PROBES:
    for conformal_method in CONFORMAL_METHODS:
        table = conformal_table(conformal_method, probe, task=TASK)
        emit_latex(
            table,
            caption=(
                f"Conformal evaluation for {conformal_method.replace('_', ' ')} "
                f"using the {probe} probe."
            ),
            label=f"tab:conformal-{conformal_method.replace('_', '-')}-{probe}",
        )

## Raw rows

Use these frames for follow-up analyses without reloading files. Source paths are retained for provenance.

In [ ]:
display(classification.head())
display(uncertainty.head())
display(conformal.head())